## Event Hub consumer - bronze (production)
Uses Event Hubs' built-in Kafka-compatible endpoint via Spark's native `kafka` format - no third-party Maven library needed, so no allowlist approval required on the shared cluster.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

catalog = "dbr_dev_ua5816bd"
login = "lena066636"
scope_name = f"{login}-scope"

connection_str = dbutils.secrets.get(scope=scope_name, key="eventhub-connection-string")
eventhub_name = dbutils.secrets.get(scope=scope_name, key="eventhub-name")
eventhub_namespace = dbutils.secrets.get(scope=scope_name, key="eventhub-namespace")

checkpoint_path = f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/_checkpoints/eventhub_orders"
bronze_table = f"{catalog}.{login}_bronze.orders_eventhub"


In [0]:
bootstrap_servers = f"{eventhub_namespace}.servicebus.windows.net:9093"

jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{connection_str}";'
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": eventhub_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": jaas_config,
    "startingOffsets": "earliest",
}


In [0]:
schema = (StructType()
    .add("order_id", LongType())
    .add("customer", StringType())
    .add("amount", DoubleType())
    .add("ts", DoubleType()))


In [0]:
df_raw = spark.readStream.format("kafka").options(**kafka_options).load()


In [0]:
df_parsed = (df_raw
    .withColumn("json_str", F.col("value").cast("string"))
    .withColumn("data", F.from_json(F.col("json_str"), schema))
    .select("data.*", F.col("timestamp").alias("enqueued_time"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
    .withColumn("source", F.lit("eventhub")))


In [0]:
(df_parsed.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table))
